In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
OUT_ROOT = Path(r"D:\SemanticBiods\data")
LANGUAGES = ["English", "French"]
RUNS = ["main", "content", "window"]
OUTCOMES = ["RSC", "Turnover"]
MODELS = {
    "M1_Frequency": ["Frequency"],
    "M2_History": ["Frequency", "PastChange"],
    "M3_Global": ["Frequency", "PastChange", "Global"],
    "M4_Local": ["Frequency", "PastChange", "Local"],
    "M5_Full": ["Frequency", "PastChange", "Global", "Local"],
}
SPECS = {
    "main": dict(lags=[0, 1, 2, 3, 4, 5], period=False),
    "lag1plus": dict(lags=[1, 2, 3, 4, 5], period=False),
    "period": dict(lags=[0, 1, 2, 3, 4, 5], period=True),
}
ALPHAS = np.logspace(-3, 5, 33)
N_OUTER, N_INNER = 5, 4
STEP = 10

def standardise(X, mean=None, sd=None):
    if mean is None:
        mean = X.mean(0)
        sd = np.where(X.std(0) < 1e-12, 1.0, X.std(0))
    return (X - mean) / sd, mean, sd


def ridge_path(Z, yc, alphas):
    """Coefficients for every alpha, on standardised Z and centred yc."""
    lam, Q = np.linalg.eigh(Z.T @ Z)
    Qb = Q.T @ (Z.T @ yc)
    return (Q @ (Qb[:, None] / (lam[:, None] + alphas[None, :]))).T


def fit_predict(Xtr, ytr, Xte, alphas):
    Ztr, mean, sd = standardise(Xtr)
    ybar = ytr.mean()
    B = ridge_path(Ztr, ytr - ybar, alphas)
    return ((Xte - mean) / sd) @ B.T + ybar


def nested_cv(X, y, words, alphas, outer_groups=None):
    """Out-of-fold predictions, penalty chosen inside each training fold."""
    og = words if outer_groups is None else np.asarray(outer_groups)
    n_splits = N_OUTER if outer_groups is None else len(np.unique(og))

    oof = np.full(len(y), np.nan)
    chosen = []
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, og):
        sse = np.zeros(len(alphas))
        for itr, ite in GroupKFold(N_INNER).split(X[tr], y[tr], words[tr]):
            pred = fit_predict(X[tr][itr], y[tr][itr], X[tr][ite], alphas)
            sse += ((pred - y[tr][ite][:, None]) ** 2).sum(0)
        best = int(np.argmin(sse))
        oof[te] = fit_predict(X[tr], y[tr], X[te], alphas)[:, best]
        chosen.append(alphas[best])
    return oof, np.asarray(chosen)


def select_alpha(X, y, words, alphas):
    sse = np.zeros(len(alphas))
    for tr, te in GroupKFold(N_OUTER).split(X, y, words):
        pred = fit_predict(X[tr], y[tr], X[te], alphas)
        sse += ((pred - y[te][:, None]) ** 2).sum(0)
    return float(alphas[int(np.argmin(sse))])


def r2(y, pred):
    return float(1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2))


def demean_within(values, keys):
    s = pd.Series(np.asarray(values, dtype=float))
    return (s - s.groupby(pd.Series(np.asarray(keys))).transform("mean")).to_numpy()


def prepare(df, lags, period):
    cols = [f"{f}_lag{l}" for f in ["Frequency", "Global", "Local",
                                    "PastChange"] for l in lags]
    X = df[cols].to_numpy(float)
    y = df["Outcome"].to_numpy(float)
    key = df["transition_start"].to_numpy()

    if period:
        y = demean_within(y, key)
        X = np.column_stack([demean_within(X[:, j], key)
                             for j in range(X.shape[1])])

    y = (y - y.mean()) / y.std()
    return X, y, cols


def main():
    for language in LANGUAGES:
        for run in RUNS:
            out = OUT_ROOT / language / run
            perf, coefs = [], []
            t0 = time.time()

            for outcome in OUTCOMES:
                df = pd.read_csv(out / f"lagged_{outcome}.csv.gz")
                words = df["word"].to_numpy()
                decade = df["transition_start"].to_numpy()

                for spec_name, spec in SPECS.items():
                    X_all, y, all_cols = prepare(df, spec["lags"],
                                                 spec["period"])
                    where = {c: i for i, c in enumerate(all_cols)}
                    store = {}

                    for model, feats in MODELS.items():
                        cols = [f"{f}_lag{l}" for f in feats
                                for l in spec["lags"]]
                        X = X_all[:, [where[c] for c in cols]]

                        for scheme, groups in (("word", None),
                                               ("decade", decade)):
                            oof, chosen = nested_cv(X, y, words, ALPHAS,
                                                    outer_groups=groups)
                            perf.append(dict(
                                language=language, run=run, spec=spec_name,
                                outcome=outcome, model=model, cv=scheme,
                                R2=r2(y, oof),
                                r=float(np.corrcoef(y, oof)[0, 1]),
                                RMSE=float(np.sqrt(np.mean((y - oof) ** 2))),
                                median_alpha=float(np.median(chosen)),
                                n_rows=len(y),
                                n_words=int(pd.unique(words).size)))
                            if scheme == "word":
                                store[model] = oof

                        alpha = select_alpha(X, y, words, ALPHAS)
                        Z, _, _ = standardise(X)
                        beta = ridge_path(Z, y - y.mean(),
                                          np.array([alpha]))[0]
                        for j, col in enumerate(cols):
                            feat, lag = col.rsplit("_lag", 1)
                            coefs.append(dict(
                                language=language, run=run, spec=spec_name,
                                outcome=outcome, model=model, feature=feat,
                                lag_decades=int(lag),
                                lag_years=int(lag) * STEP,
                                beta=float(beta[j]), alpha=alpha,
                                marginal_r=float(
                                    np.corrcoef(X[:, j], y)[0, 1])))

                    pd.DataFrame({"word": words, "transition_start": decade,
                                  "observed": y,
                                  **{f"pred_{m}": v
                                     for m, v in store.items()}}).to_csv(
                        out / f"oof_{spec_name}_{outcome}.csv.gz",
                        index=False, compression="gzip")
                    print(f"[{language}/{run}] {outcome} {spec_name} done",
                          flush=True)

            pd.DataFrame(perf).to_csv(out / "model_performance.csv",
                                      index=False)
            pd.DataFrame(coefs).to_csv(out / "trf_coefficients.csv",
                                       index=False)
            print(f"  -> {(time.time() - t0) / 60:.1f} min\n", flush=True)


if __name__ == "__main__":
    main()


[English/main] RSC main done
[English/main] RSC lag1plus done
[English/main] RSC period done
[English/main] Turnover main done
[English/main] Turnover lag1plus done
[English/main] Turnover period done
  -> 2.2 min

[English/content] RSC main done
[English/content] RSC lag1plus done
[English/content] RSC period done
[English/content] Turnover main done
[English/content] Turnover lag1plus done
[English/content] Turnover period done
  -> 1.6 min

[English/window] RSC main done
[English/window] RSC lag1plus done
[English/window] RSC period done
[English/window] Turnover main done
[English/window] Turnover lag1plus done
[English/window] Turnover period done
  -> 1.0 min

[French/main] RSC main done
[French/main] RSC lag1plus done
[French/main] RSC period done
[French/main] Turnover main done
[French/main] Turnover lag1plus done
[French/main] Turnover period done
  -> 2.3 min

[French/content] RSC main done
[French/content] RSC lag1plus done
[French/content] RSC period done
[French/content] 